# LifeMate fine-tune LLM (Kaggle)

In [ ]:
# Code comes from the private Kaggle Dataset holding the pack_code.sh zip (auto-extracted by Kaggle).
# Attach it plus the competition data dataset. Needs a GPU session (T4 x2 or P100) and Internet ON for pip + HF.
!cp -r /kaggle/input/lifemate-code/. /kaggle/working/lifemate && ls /kaggle/working/lifemate
%cd /kaggle/working/lifemate

In [ ]:
%pip install -q -r ops/requirements/notebook.txt -r ops/requirements/train.txt pytest
# vllm is only needed for the agent-eval stage (llm pipeline "eval"); pinned from uv.lock (0.32.0).
!nvidia-smi > /dev/null 2>&1 && pip install -q vllm==0.32.0 || echo "no GPU: skipping vllm"
# ASR engine for the ASR cache below (configs/v0.toml uses faster-whisper large-v3)
%pip install -q faster-whisper==1.2.1


In [ ]:
!python -c "from pathlib import Path; from lifemate.core.envinfo import detect; print(detect(Path('.')))"

In [ ]:
!python -m pytest -q tests/train -m "not data and not tokenizer"

In [ ]:
# Resume: if a previous session saved runs/ as a Kaggle Dataset version, restore it so
# pipeline_progress.json and checkpoints are found and the pipeline continues from the last finished stage.
import os, shutil
src = "/kaggle/input/lifemate-runs"
if os.path.isdir(src):
    shutil.copytree(src, "/kaggle/working/runs", dirs_exist_ok=True)
    print("restored", os.listdir("/kaggle/working/runs"))
else:
    print("no previous runs dataset attached; starting fresh")

In [ ]:
# ASR cache (SPEC 6.2.1): configs/train/llm_v1.toml trains on 70% ASR text, read from data/asr/train_asr.json.
# Built once (slow: ASR on every train/val sample); reused from a restored runs folder when present.
import os, shutil, subprocess
cache = "data/asr/train_asr.json"
saved = '/kaggle/working/runs/asr/train_asr.json'
if saved and os.path.isfile(saved) and not os.path.isfile(cache):
    os.makedirs("data/asr", exist_ok=True); shutil.copy(saved, cache); shutil.copy(saved + ".meta.json", cache + ".meta.json")
if not os.path.isfile(cache):
    subprocess.run(["python", "-m", "lifemate.train.asr_cache", "--config", "configs/v0.toml", "--env", "kaggle",
                    "--out", cache, "--ids", "configs/splits/train.txt", "configs/splits/val_seen.txt",
                    "configs/splits/val_unseen.txt"], check=True)
    if saved:
        os.makedirs(os.path.dirname(saved), exist_ok=True); shutil.copy(cache, saved); shutil.copy(cache + ".meta.json", saved + ".meta.json")
print("ASR cache:", cache, os.path.getsize(cache), "bytes")


In [ ]:
# One command (SPEC 6.2.6): data -> train -> select -> merge -> eval -> choose -> register.
# Add --from-stage/--to-stage to run part of it; progress is in <work>/pipeline_progress.json.
# --limit 30 keeps the agent eval short; raise it for the final choice.
!python -m lifemate.train.llm_pipeline --config configs/train/llm_v1.toml --env kaggle --limit 30

In [ ]:
!cat artifacts/models/registry.csv
!find /kaggle/working/runs/train -maxdepth 3 -name "metrics.jsonl" -o -maxdepth 3 -name "train_config.json" | head

In [ ]:
# Persist runs/ (checkpoints, merged model, metrics) as a new version of the private dataset "lifemate-runs";
# create it once as described in ops/notebooks/kaggle_run.ipynb. Attach that dataset to the next session to resume.
!kaggle datasets version -p /kaggle/working/runs -m "train $(cat VERSION)" --dir-mode zip